# TF-IDF Calculation Changes

This notebook documents the minimal changes made to replace the arbitrary `0.75` threshold in the TF-IDF calculation with a mathematically sound proportion-based approach.

## Problem Statement

The original code used a binary threshold to determine if an answer was "mostly yes" or "mostly no":

```python
# Original approach
mostly_yes_count: int = 1 if answer.yes_count * 0.75 > answer.no_count else 0
mostly_no_count: int = 1 if answer.no_count * 0.75 > answer.yes_count else 0
```

**Issue**: The `0.75` value was arbitrary and did not reflect the actual strength of the answer.

### Example Problem
For Donald Trump:
- "Is the character famous?" → Almost everyone answers YES (common question)
- "Is the character a president?" → Fewer people answer YES (rare, discriminative question)

With the old approach, both could result in the same binary `1` value, losing the nuance that "president" is a more discriminative question.

## Solution: Proportion-Based IDF

Instead of counting binary yes/no answers, we now use the **actual proportion** of yes/no votes for each answer, then compute the **average proportion** across all characters to determine IDF.

### Mathematical Formulation

For each answer (character-question pair):
$$\text{yes\_prop} = \frac{\text{yes\_count}}{\text{yes\_count} + \text{no\_count}}$$

For each question, we compute the average proportion:
$$\text{avg\_yes} = \frac{\sum \text{yes\_prop}}{N}$$

Then the IDF becomes:
$$\text{IDF} = \log_2\left(\frac{1}{\text{avg\_yes} + \epsilon}\right)$$

Where $\epsilon = 10^{-10}$ prevents division by zero.

### Key Insight
- **Common answers** (high avg_prop) → **Low IDF** (less discriminative)
- **Rare answers** (low avg_prop) → **High IDF** (more discriminative)

## Files Modified

Only **2 files** were changed:

### 1. `src/app/models/dataclasses.py`
Changed `AnswerStat` field types from `int` to `float` to store proportions:

```diff
@dataclass
class AnswerStat:
    count: int
-   mostly_yes_count: int
-   mostly_no_count: int
+   mostly_yes_count: float
+   mostly_no_count: float
    answers: list[Answer] = field(default_factory=list)
```

### 2. `src/app/utilities/utilities.py`
Modified the `indexation()` function in two places:

## Code Changes in Detail

### Change 1: Accumulation Loop (collecting proportions instead of binary counts)

**Before:**
```python
for answer in Answer.objects.iterator():
    count: int = 1 if answer.yes_count + answer.no_count + answer.pass_count > 0 else 0
    # Binary threshold approach
    mostly_yes_count: int = 1 if answer.yes_count * 0.75 > answer.no_count else 0
    mostly_no_count: int = 1 if answer.no_count * 0.75 > answer.yes_count else 0

    answers_dict[answer.question_id] += AnswerStat(count, mostly_yes_count, mostly_no_count, [answer])
```

**After:**
```python
for answer in Answer.objects.iterator():
    count: int = 1 if answer.yes_count + answer.no_count + answer.pass_count > 0 else 0
    # Proportion-based approach
    total = answer.yes_count + answer.no_count
    yes_prop: float = answer.yes_count / total if total > 0 else 0
    no_prop: float = answer.no_count / total if total > 0 else 0

    answers_dict[answer.question_id] += AnswerStat(count, yes_prop, no_prop, [answer])
```

### Change 2: IDF Calculation (using average proportions)

**Before:**
```python
for answer_stat in answers_dict.values():
    yes_idf: float = log((answer_stat.count) / (answer_stat.mostly_yes_count + 1), 2)
    no_idf: float = log((answer_stat.count) / (answer_stat.mostly_no_count + 1), 2)
```

**After:**
```python
for answer_stat in answers_dict.values():
    # IDF based on average proportion rarity
    avg_yes = answer_stat.mostly_yes_count / answer_stat.count if answer_stat.count > 0 else 0
    avg_no = answer_stat.mostly_no_count / answer_stat.count if answer_stat.count > 0 else 0
    yes_idf: float = log(1 / (avg_yes + 1e-10), 2) if avg_yes < 1 else 0
    no_idf: float = log(1 / (avg_no + 1e-10), 2) if avg_no < 1 else 0
    yes_idf = max(0, yes_idf)
    no_idf = max(0, no_idf)
```

## Numerical Example

Consider two questions applied to all characters in the database:

| Question | avg_yes_prop | IDF Calculation | IDF Value |
|----------|--------------|-----------------|-----------|
| "Is the character famous?" | 0.85 | $\log_2(1/0.85) = 0.23$ | **Low** |
| "Is the character a president?" | 0.02 | $\log_2(1/0.02) = 5.64$ | **High** |

This correctly gives "president" a much higher IDF than "famous", making it more discriminative for identifying specific characters like Donald Trump.

In [ ]:
# Interactive demonstration of the IDF formula
from math import log

def calculate_idf(avg_proportion: float) -> float:
    """Calculate IDF from average yes/no proportion."""
    epsilon = 1e-10
    if avg_proportion >= 1:
        return 0
    idf = log(1 / (avg_proportion + epsilon), 2)
    return max(0, idf)

# Example proportions
examples = [
    ("Famous", 0.85),
    ("Human", 0.70),
    ("Male", 0.55),
    ("Politician", 0.15),
    ("President", 0.02),
    ("French President", 0.005),
]

print("Question".ljust(20), "Avg Proportion", "IDF")
print("-" * 50)
for name, prop in examples:
    idf = calculate_idf(prop)
    print(f"{name.ljust(20)} {prop:.3f}          {idf:.2f}")

## Summary of Changes

| Aspect | Before | After |
|--------|--------|-------|
| **Threshold** | Arbitrary `0.75` | None (uses proportions) |
| **Data type** | Binary (`int`: 0 or 1) | Continuous (`float`: 0 to 1) |
| **IDF formula** | $\log_2(\frac{N}{\text{count} + 1})$ | $\log_2(\frac{1}{\text{avg\_prop} + \epsilon})$ |
| **Files changed** | - | 2 files |
| **Lines changed** | - | ~15 lines |

### Benefits
1. ✅ **No arbitrary threshold** - Uses actual vote proportions
2. ✅ **Preserves nuance** - 90% yes ≠ 60% yes
3. ✅ **Correct IDF ordering** - Rare questions get higher weight
4. ✅ **Minimal changes** - Only modified the indexation logic